# T08 — ConvNeXt-Tiny: kết hợp sau ablation
Chỉ chọn T08 khi T00–T07 đã hoàn tất và được audit. Chọn hai yếu tố khác nhau theo validation.
Nếu thiếu hai yếu tố cải thiện vượt ngưỡng, tổ hợp được ghi là thăm dò.
Giữ recipe T08 đã khóa khi resume. Dùng kernel `.venv`; không chạy đồng thời với master.
Muốn chọn thủ công một tổ hợp có lý do: dùng CLI `--t08` theo hướng dẫn trong temp.

## 1. Môi trường

In [ ]:
import os, sys, json
from pathlib import Path
from dataclasses import asdict
ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo")
CODE = ROOT / "submissions/2A202602810_NguyenThaiAnh/code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
import ablation_notebook as ab
import backbone_notebook as nbtrain
import train as trainer
assert Path(trainer.__file__).resolve() == CODE / "train.py"
print("ROOT:", ROOT)
print("Python:", sys.executable)

## 2. Kiểm tra và đọc các thí nghiệm đơn

In [ ]:
RUN_TRAINING = True
RUN_T08 = True
MIN_DELTA = .002

plan = ab.active_plan()
configs = ab.configs_for(plan)
base = configs["T00"]
summary, baseline_history, metrics = ab.verify_ablation(base, base)
print(f"T00 = B03/seed{base.seed}; F1 val = {metrics['macro_f1']:.4%}; đủ {base.epochs} epoch")
display(pd.DataFrame([{"exp_id":s.exp_id, "axis":s.axis, "changes":s.changes,
                       "status":nbtrain.training_status(configs[s.exp_id])} for s in plan]))
print("Không tái train B03; mọi run T kế thừa seed/split/batch/LR/scheduler/preprocessing của B03.")

frame = ab.collect_results(require_complete=False)
display(frame[["exp_id", "status", "macro_f1_val", "delta_macro_f1_pp"]])

## 3. Khóa tổ hợp và train

In [ ]:
complete = set(frame.loc[frame["status"] == "complete", "exp_id"])
required = {f"T{i:02}" for i in range(8)}
decision = None
if required <= complete:
    saved = ab.ABLATION_OUTPUT / "t08_selection.json"
    decision = json.loads(saved.read_text()) if saved.exists() else ab.propose_combination(frame, min_delta=MIN_DELTA)
    display(decision)
    print("Ngưỡng thực dụng 0.002 = 0.2 điểm % F1; không phải kiểm định ý nghĩa thống kê.")
    if RUN_TRAINING and RUN_T08:
        decision = ab.lock_combination(decision)
        cfg08 = ab.configs_for(ab.active_plan())["T08"]
        nbtrain.run_in_subprocess(cfg08, planned_group=4)
        ab.verify_ablation(cfg08, base)
        print("T08 Output audit: PASS")
else:
    print("Hoàn thành các run trước T08:", sorted(required - complete))

## 4. Tổng hợp và biểu đồ

In [ ]:
frame = ab.collect_results(require_complete=False)
display(frame[["exp_id", "axis", "status", "macro_f1_val", "top1_val", "nll_val", "delta_macro_f1_pp"]])
print("Δ tính bằng điểm phần trăm; new/resume chưa có metric kết quả đầy đủ.")
print("Bảng:", ab.ABLATION_OUTPUT / "training_ablation.csv")


fig = ab.plot_results(frame)
plt.show()
plt.close(fig)
class_table = pd.read_csv(ab.ABLATION_OUTPUT / "training_per_class_val.csv")
display(class_table.pivot(index="class_name", columns="exp_id", values=["f1", "recall"]))
print("Số ảnh từng lớp lấy từ toàn bộ validation; không thay bằng subset.")

## 5. Kiểm tra hoàn thành phần 8

In [ ]:
selection = json.loads((ab.ABLATION_OUTPUT / "screening_selection.json").read_text())
display(selection)
display(Markdown((ab.ABLATION_OUTPUT / "ablation_report.md").read_text()))